In [61]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, f1_score, accuracy_score
from tensorflow.keras.utils import to_categorical

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input, LSTM
from sklearn.utils import class_weight

import seaborn as sns
import matplotlib.pyplot as plt

from tensorflow.keras.callbacks import EarlyStopping
from scipy.stats import pearsonr

from functions.binary import *
from functions.report import *

from fpdf import FPDF
import os
import ta

from collections import defaultdict

import joblib
import MetaTrader5 as mt5
import datetime
import time
import pytz

In [62]:
data = pd.read_csv('training_data/eurusd_dataset_2025_binary.csv')
data = data.drop(['Date_Time'], axis=1)

X = data.drop(['label'], axis=1)
y = data['label'].astype(int)

In [63]:
# Initialize
X_full = X.copy()
X_full['pred'] = np.nan
X_full['confidence'] = np.nan
X_full['trade_pnl'] = 0.0
X_full['lot_size'] = 0.0
X_full['trade_open'] = False
X_full['balance'] = np.nan

# Parameters
sequence_length = 12
window_size = 5000
val_size = 1000
step = 1000
cost_per_trade = 1.5 # review this, it can be changed to be in euros    
pip_value_per_standard_lot = 10
initial_account_balance = 10000.0
risk_per_trade_percentage = 0.01
steps = 12 # ver se isto influencia, meti todas as funções dependentes desta variável
extra_steps = 0
class_to_direction = {0: -1, 1: 0, 2: 1}
threshold = 0.7 #testar diferentes

# Tracking
current_account_balance = initial_account_balance
f1_per_window, acc_per_window, profit_per_window, trade_per_window, window_indices = [], [], [], [], []
vol_per_window, local_vol_per_window, losing_profit, winning_profit = [], [], [], []
profit_per_class = defaultdict(float)
trades_per_class = defaultdict(int)
winning_trades, losing_trades = 0, 0

for w, start in enumerate(range(0, len(X_full) - window_size - val_size - steps, step)):
    print(f"\n Window {w}")

    # Split
    train_X = X_full.iloc[start : start + window_size].drop(columns=['pred', 'confidence', 'trade_pnl', 'lot_size', 'trade_open', 'balance'])
    train_y = y[start : start + window_size]

    val_X = X_full.iloc[start + window_size : start + window_size + val_size].drop(columns=['pred', 'confidence', 'trade_pnl', 'lot_size', 'trade_open', 'balance'])
    val_y = y[start + window_size : start + window_size + val_size]

    # Standardize
    scaler = StandardScaler().fit(train_X)
    train_X_scaled = scaler.transform(train_X)
    val_X_scaled = scaler.transform(val_X)

    train_X_seq, train_y_seq = create_lstm_sequences(train_X_scaled, train_y.values, sequence_length)
    val_X_seq, val_y_seq = create_lstm_sequences(val_X_scaled, val_y.values, sequence_length)

    train_y_cat = to_categorical(train_y_seq, num_classes=3)
    val_y_cat = to_categorical(val_y_seq, num_classes=3)

    # Build & train model
    input_features = train_X_seq.shape[2]
    sequence_length = train_X_seq.shape[1]
    model = build_model_rnn(sequence_length, input_features, num_classes=3)
    cw = dict(enumerate(class_weight.compute_class_weight(class_weight='balanced', classes=np.unique(train_y_seq), y=train_y_seq)))
    model.fit(train_X_seq, train_y_cat, epochs=30, batch_size=32, class_weight=cw, verbose=0)

    # Predict
    #Prevê a melhor class e guarda a confiança
    preds_probs = model.predict(val_X_seq, verbose=0)
    preds = np.argmax(preds_probs, axis=1)
    confidences = np.max(preds_probs, axis=1)

    val_start = start + window_size
    adjusted_start = val_start + sequence_length
    max_len = min(len(preds), len(X_full) - adjusted_start)
    pred_indices = range(adjusted_start, adjusted_start + len(preds))

    #armazena na df
    X_full.loc[pred_indices, 'pred'] = preds[:max_len]
    X_full.loc[pred_indices, 'confidence'] = confidences[:max_len]


    # Optimize SL/TP
    sl_tp_map = optimize_sl_tp_per_class(
        y=train_y,
        close_prices=X_full['Open'].iloc[start:start + window_size].values,
        highs=X_full['High'].iloc[start:start + window_size].values,
        lows=X_full['Low'].iloc[start:start + window_size].values,
        sl_values=[8, 10, 12, 15, 20],
        tp_values=[10, 12, 15, 20, 25],
        class_to_direction=class_to_direction,
        cost_per_trade=cost_per_trade,
        steps=steps
    )
    
    avg_duration_by_class = estimate_avg_duration_per_class(
        y=train_y,
        close_prices=X_full['Open'].iloc[start:start + window_size].values,
        highs=X_full['High'].iloc[start:start + window_size].values,
        lows=X_full['Low'].iloc[start:start + window_size].values,
        sl_tp_map=sl_tp_map,
        class_to_direction=class_to_direction,
        steps=steps
    )

    # Trade Simulation
    min_broker_lot_size = 0.01
    max_broker_lot_size = 50.0
    profit = 0.0
    trades = 0
    last_trade_close_idx = -1

    for i, t in enumerate(pred_indices):
        if np.isnan(X_full.at[t, 'pred']):
            continue
        if t < last_trade_close_idx:
            continue

        pred = int(X_full.at[t, 'pred'])
        direction = class_to_direction.get(pred, 0)
        if direction == 0:
            continue

        sltp = sl_tp_map.get(pred, {'sl': None, 'tp': None})
        if sltp['sl'] is None or sltp['tp'] is None or sltp['sl'] <= 0:
            continue

        confidence = X_full.at[t, 'confidence']
        if confidence < threshold:
            continue

        monetary_risk = current_account_balance * risk_per_trade_percentage
        lot_size = monetary_risk / (sltp['sl'] * pip_value_per_standard_lot)
        lot_size = round(min(max(lot_size, min_broker_lot_size), max_broker_lot_size), 2)
        if lot_size < 0.01:
            continue

        entry = X_full.at[t, 'Open']
        highs_seq = X_full['High'].iloc[t : t + steps].values
        lows_seq = X_full['Low'].iloc[t : t + steps].values
        limit = avg_duration_by_class.get(pred, 0) + extra_steps

        result_pips, rel_exit_idx = simulate_trade(entry, highs_seq[:limit], lows_seq[:limit], direction, sltp['sl'], sltp['tp'])
        result_pips -= cost_per_trade

        trade_profit = result_pips * pip_value_per_standard_lot * lot_size
        profit += trade_profit
        trades += 1
        current_account_balance += trade_profit
        last_trade_close_idx = t + rel_exit_idx

        X_full.at[t, 'trade_open'] = True
        X_full.at[t, 'trade_pnl'] = trade_profit
        X_full.at[t, 'lot_size'] = lot_size
        X_full.at[t, 'balance'] = current_account_balance

        profit_per_class[pred] += trade_profit
        trades_per_class[pred] += 1
        if result_pips > 0:
            winning_trades += 1
            winning_profit.append(trade_profit)
        elif result_pips < 0:
            losing_trades += 1
            losing_profit.append(trade_profit)

    # Track metrics
    vol = val_X['Close'].pct_change()
    vol_std = vol.std()
    vol_local = val_X['vol_local'].mean()

    f1 = f1_score(val_y.iloc[:max_len], preds[:max_len], average='weighted')
    acc = accuracy_score(val_y.iloc[:max_len], preds[:max_len])
    f1_per_window.append(f1)
    acc_per_window.append(acc)
    profit_per_window.append(profit)
    trade_per_window.append(trades)
    window_indices.append(w)
    vol_per_window.append(vol_std)
    local_vol_per_window.append(vol_local)

    print(f"Profit: {profit:.2f}, Trades: {trades}, F1: {f1:.3f}, Acc: {acc:.3f}, Balance: {current_account_balance:.2f}")


 Window 0
Profit: 266.66, Trades: 39, F1: 0.506, Acc: 0.456, Balance: 10266.66

 Window 1
Profit: -507.00, Trades: 46, F1: 0.470, Acc: 0.476, Balance: 9759.66

 Window 2
Profit: 392.19, Trades: 25, F1: 0.547, Acc: 0.486, Balance: 10151.85

 Window 3
Profit: -331.51, Trades: 26, F1: 0.608, Acc: 0.563, Balance: 9820.34

 Window 4
Profit: 407.77, Trades: 31, F1: 0.591, Acc: 0.539, Balance: 10228.11

 Window 5
Profit: 170.74, Trades: 17, F1: 0.603, Acc: 0.577, Balance: 10398.85

 Window 6
Profit: 295.92, Trades: 42, F1: 0.540, Acc: 0.499, Balance: 10694.77

 Window 7
Profit: -1416.13, Trades: 69, F1: 0.349, Acc: 0.349, Balance: 9278.64

 Window 8


KeyboardInterrupt: 

In [20]:
X_full.to_csv('eurusd_predictions_2025_binary.csv', index=False)

In [65]:
generate_model_report_pdf(
    steps,
    extra_steps,
    window_indices,
    f1_per_window,
    acc_per_window,
    profit_per_window,
    trade_per_window,
    vol_per_window,
    local_vol_per_window,
    losing_profit,
    winning_profit,
    initial_account_balance,
    window_size,
    val_size,
    step,
    cost_per_trade,
    pip_value_per_standard_lot,
    risk_per_trade_percentage,
    winning_trades,
    losing_trades,
    profit_per_class,
    trades_per_class,
    threshold,
    report_filename="model_eurusd_binary_6k.pdf"
)

AttributeError: 'list' object has no attribute 'mean'